# Imports

In [ ]:
%load_ext autoreload
%autoreload 2
import json
import os 
import sys
sys.path.append("../src")

import matplotlib.pyplot as plot
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import matplotlib.dates as mdates

import plotly.io as pio
import plotly.express as px

import pandas as pd
import numpy as np
import scipy
import sklearn as sk
import ruptures as rpt
import pymannkendall as mk
from collections import Counter, defaultdict
from tqdm import tqdm
import statsmodels.api as sm
from itertools import combinations
from bs4 import BeautifulSoup
import re
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import csr_matrix
from mlxtend.frequent_patterns import apriori, association_rules
from prefixspan import PrefixSpan
from functools import reduce
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import ttest_ind

from bisect import bisect_left,bisect_right
from global_utils.graphs_utils import get_subplots, prepare_subplots, fancy_histogram, color_dic, arrange_twin_plots
from global_utils.files_utils import create_tree_folder
from ticket_analysis.ticket_utils import *
from sqlalchemy import create_engine
import dask.dataframe as dd
import duckdb
import subprocess
import pickle
pd.set_option("display.max_columns", None)
import warnings
warnings.filterwarnings("ignore")
# warnings.filterwarnings("default")

# Read Data

In [ ]:
DATA_DIR = "../Database"
fontsize = 18
if os.path.exists(DATA_DIR):
    print("Can see Data_Dir")
colors = color_dic["big"]

In [ ]:
meter_values = [pd.read_csv(os.path.join(DATA_DIR, 'MeterValues.csv')), pd.read_csv(os.path.join(DATA_DIR, "MeterValues_Extra.csv")), pd.read_csv(os.path.join(DATA_DIR, "MeterValues_Extra.csv"))]
meter_values = pd.concat(meter_values, ignore_index=True)
meter_values['MeterValueTimeStamp'] = pd.to_datetime(meter_values['MeterValueTimeStamp'], format="mixed")
meter_values['UpdatedTimeStamp'] = pd.to_datetime(meter_values['UpdatedTimeStamp'], format="mixed")

transactions = [pd.read_csv(os.path.join(DATA_DIR, "Transactions.csv")),pd.read_csv(os.path.join(DATA_DIR, "Transactions_Extra.csv")), pd.read_csv(os.path.join(DATA_DIR, "LegacyTransactions_new.csv"))]
transactions = pd.concat(transactions, ignore_index=True)
transactions["StartAt"] = pd.to_datetime(transactions["StartAt"], format="mixed")
transactions["EndAt"] = pd.to_datetime(transactions["EndAt"], format="mixed")

events = pd.read_parquet(os.path.join(DATA_DIR, "Events_Trimmed.parquet"))

In [ ]:
name_of_chargers = list(set(transactions["ChargerID"].unique()) & set(events["device_id"].unique()) & set(meter_values["ChargerID"].unique()))

In [ ]:
charger_transactions = {key: value.sort_values(by="StartAt") for key, value in transactions.groupby("ChargerID")}
charger_events = {key: value.sort_values(by="event_time") for key, value in events.groupby("device_id", observed=True)}
charger_meter_values = {key: {trId: group.sort_values(by="MeterValueTimeStamp") for trId, group in value.groupby("TransactionID")} for key, value in meter_values.groupby("ChargerID")}

In [ ]:
merge_isp = pd.read_csv(os.path.join(DATA_DIR, "Merged_ISP.csv"))
merge_isp["Date"] = pd.to_datetime(merge_isp["Date"]).dt.tz_localize("UTC")


In [ ]:
# min_max_events = pd.read_csv(os.path.join(DATA_DIR, "min_max_events.csv"))
min_max_events = events.groupby("device_id")["event_time"].agg(["min", "max"]).reset_index()
merge_isp_only_events = merge_isp.merge(
    min_max_events,
    left_on="charging_station_id",
    right_on="device_id",
    how="inner"
).drop(columns="device_id")

mask = (merge_isp_only_events["Date"]>=merge_isp_only_events["min"]) & (merge_isp_only_events["Date"]<=merge_isp_only_events["max"])
merge_isp_only_events = merge_isp_only_events[mask]
merge_isp_only_events = merge_isp_only_events.drop(columns=["min", "max"])


In [ ]:
# min_max_transactions = pd.read_csv(os.path.join(DATA_DIR, "min_max_transactions_merged.csv")).drop(columns="Unnamed: 0")
min_max_transactions = transactions.groupby("ChargerID")["StartAt"].agg(["min", "max"]).reset_index()
merge_isp_only_transactions = merge_isp.merge(
    min_max_transactions,
    left_on="charging_station_id",
    right_on="ChargerID",
    how="inner"
).drop(columns="ChargerID")

mask = (merge_isp_only_transactions["Date"]>=merge_isp_only_transactions["min"]) & (merge_isp_only_transactions["Date"]<=merge_isp_only_transactions["max"])
merge_isp_only_transactions = merge_isp_only_transactions[mask]
merge_isp_only_transactions = merge_isp_only_transactions.drop(columns=["min", "max"])


In [ ]:
chargers = np.array(merge_isp_only_events["charging_station_id"].tolist())

In [ ]:
ellipse_tickets = pd.read_csv(os.path.join(DATA_DIR, "ellipse_tickets.csv"))
ellipse_tickets["Date"] = pd.to_datetime(ellipse_tickets["Date"])

In [ ]:
filter_cols = ["charging_station_id", "Title", "Date", "Serial Number"]
merge_isp_only_events_filtered = merge_isp_only_events.merge(ellipse_tickets[filter_cols], on=filter_cols, how="left", indicator=True)
merge_isp_only_events_filtered = merge_isp_only_events_filtered[merge_isp_only_events_filtered["_merge"] == "left_only"].drop(columns="_merge")

# Exploration

## Clusters

In [ ]:
event_list = []
specific_ticket = merge_isp_only_events.iloc[0]
threshold_gap = pd.Timedelta("1W")
list_subset_events = []

index = 0
cols = ["name"]
# for _, specific_ticket in merge_isp_only_events.iterrows():
for _, specific_ticket in merge_isp_only_events_filtered.iterrows():
    specific_charger = specific_ticket["charging_station_id"]
    date = specific_ticket["Date"]

    specific_events = charger_events[specific_charger].copy()
    specific_events = specific_events[~specific_events["name"].isna()]
    specific_events = specific_events[specific_events["name"] != "VehicleId"]
    specific_events = specific_events[specific_events["name"] != "Connector Availability Update"]
    specific_events = specific_events[specific_events["name"] != "Charger Availability Update"]
    specific_events = specific_events[specific_events["severity"] != "Information"]
    specific_events["name"] = specific_events["name"].str.replace(r'\d+','X',regex=True)

    subset_events = specific_events[(specific_events["event_time"]>= date-threshold_gap) & (specific_events["event_time"]<= date)]
    subset_events["closest"] = subset_events["event_time"] - date
    subset_events = subset_events.groupby(cols, observed=True, as_index=False, dropna=False).agg(count=("event_time", "size"), device_id=("device_id", "first"), closest=("closest", lambda x: min(x, key=abs))).reset_index(drop=1)
    total_counts = specific_events.groupby(cols, observed=True, as_index=False, dropna=False).agg(count_total=("event_time", "size")).reset_index(drop=1)
    subset_events = subset_events.merge(total_counts, on=cols, how="left")
    subset_events["Rarity"] = subset_events["count"] / subset_events["count_total"]

    list_subset_events.append(subset_events)
    index += 1

In [ ]:
event_threshold= 0
event_list = [
    [(nm, rar) for nm, rar in subset_events[subset_events["count_total"]>event_threshold][["name","count"]].values]
    for subset_events in list_subset_events
]

total_event_list = [
    [(nm, rar) for nm, rar in subset_events[subset_events["count_total"]>event_threshold][["name","count_total"]].values]
    for subset_events in list_subset_events
]


max_window = []
for ch in merge_isp_only_events_filtered["charging_station_id"]:
    specific_events = charger_events[ch]
    max_time, min_time = specific_events["event_time"].max(), specific_events["event_time"].min()
    max_window.append((max_time - min_time).total_seconds()/(3600*24))
max_window = np.array(max_window)

In [ ]:
all_names = sorted({name for arr in event_list for name, _ in arr})
name_idx = {n: i for i, n in enumerate(all_names)}

X = np.zeros((len(event_list), len(all_names)))
X_total = np.zeros((len(total_event_list), len(all_names)))
for i, arr in enumerate(event_list):
    for name, rar in arr:
        X[i, name_idx[name]] = rar
for i, arr in enumerate(total_event_list):
    for name, rar in arr:
        X_total[i, name_idx[name]] = rar
# X_norm = sk.preprocessing.normalize(X)
X_norm = X - X.mean(axis=0)
X_norm = sk.preprocessing.StandardScaler().fit_transform(X)
tdidf = sk.feature_extraction.text.TfidfTransformer(norm='l2', use_idf=True, smooth_idf=True)
X_norm = tdidf.fit_transform(X).toarray()


In [ ]:
titles = np.array(list(merge_isp_only_events_filtered["Title"].str.lower().str.extract(r"\]\s*(.+)$").fillna("").values[:,0]))
vectorizer = sk.feature_extraction.text.TfidfVectorizer(stop_words="english")
titles_emb = vectorizer.fit_transform(titles)
vocab = vectorizer.get_feature_names_out()

In [ ]:
pca_2d = sk.decomposition.PCA(n_components=2)
tsne_2d = sk.manifold.TSNE(n_components=2, random_state=42)
hdb = sk.cluster.HDBSCAN(min_samples=10, metric="euclidean")

In [ ]:
tsne_events = tsne_2d.fit_transform(X_norm)
pca_events = pca_2d.fit_transform(X_norm)
tsne_titles = tsne_2d.fit_transform(titles_emb.toarray())
pca_titles = pca_2d.fit_transform(titles_emb.toarray())

## Using self-made categories

In [ ]:
CATEGORIES = {
    "Connectivity / Offline": [
        "offline", "ocpp", "scb offline", "not reachable", "connectivity",
        "cu not reachable", "cms", "onboarding", "modbus", "secc", "sec pilot",
        "can comm", "no scb connection", "coap", "communication issue",
        "preauthorize", "station status", "wan port", "lan cable",
        "network issue", "ip address",
    ],
    "Charging Failure": [
        "charging issue", "charging station issue", "unable to charge",
        "charger out of order", "out of order", "out of service",
        "charger not working", "charger unavailable", "alloutletsunavailable",
        "outlets out of order", "outlet out of order", "outlet 1", "outlet 2",
        "charger issue", "charger error", "charger malfunction", "charger ooo",
        "charger faulty", "charging interruption", "charging session",
        "charging not possible", "ghost session", "outlets unavailable",
        "outlet unavailable", "all outlets ooo", "charging point issue",
        "locked gun", "gun issue", "ev not charging", "session issue",
        "charger repair", "charging station repair", "not charging",
        "station not available", "station repair",
    ],
    "Display / HMI": [
        "screen", "display", "hmi", "blackscreen", "black screen", "scramble",
        "delamination", "detached", "qr", "screensaver", "stuck on",
        "blistering", "hmi issue", "hmi error", "blank screen",
    ],
    "Power / Converter": [
        "converter", "power converter", "power loss", "powerloss",
        "missing converter", "contactor", "mainbreaker", "spd", "fuse",
        "coils", "failsafe", "mainfan", "lem issue", "internal error",
        "hardware issue", "overheat", "fan error", "automation fan",
        "thermal", "overcurrent", "undervoltage", "overvoltage",
    ],
    "Cable / Connector": [
        "cable", "transit damage", "connector issue", "plug failure",
        "faulty plug", "cable theft", "cable temp", "cable damaged",
        "holster", "inlet",
    ],
    "Firmware / Software Update": [
        "firmware", "fw", "rollout", "configuration", "config",
        "screensaver update", "update screensaver", "downgrade",
        "commissioning", "software update", "sw update", "flash",
        "cfast", "reboot", "reset",
    ],
    "Log Investigation": [
        "log", "high volume of log", "download logs", "unable to download",
    ],
    "Payment Terminal": [
        # "payment terminal", "terminal candidate", "rfid", "payment",
        # "terminal issue", "pt terminal", "ccv",
        "payment", "payment terminal", "pay", "card", "rfidcase"
    ],
    # "Frequency / Time Sync": [
    #     "frequency", "time sync", "time zone", "ntp", "clock",
    # ],
    # "Technical Query / Other": [],  # catch-all
}


In [ ]:
classified= classify_tickets(merge_isp)
classified_events = classify_tickets(merge_isp_only_events)
classified_events_filtered = classify_tickets(merge_isp_only_events_filtered)
classified_transactions = classify_tickets(merge_isp_only_transactions)

In [ ]:
labels_self = classified_events_filtered["category"].values
unique_labels_self = sorted(set(labels_self))

In [ ]:
ellipse_mask = (-14<tsne_events[:,0]) & (tsne_events[:,0]<=0) & (-24<tsne_events[:,1]) & (tsne_events[:,1]<-9)

In [ ]:
ellipse_tickets = merge_isp_only_events.iloc[ellipse_mask]
non_ellipse_tickets = merge_isp_only_events.iloc[~ellipse_mask]
ellipse_tickets = ellipse_tickets.merge(
    min_max_events,
    left_on = "charging_station_id",
    right_on="device_id"
).drop(columns="device_id")
ellipse_tickets["parsed_html"] = ellipse_tickets["HTML Description"].apply(_extract_html_issue)

X_ellipse = X[ellipse_mask]
X_non_ellipse = X[~ellipse_mask]

## Automatic by events

In [ ]:
labels_events = hdb.fit_predict(X_norm)
n_clusters = len(set(labels_events)) - (1 if -1 in labels_events else 0)
n_noise = (labels_events == -1).sum()
print(f"Found {n_clusters} clusters, and {n_noise} noisy points")

## Automatic by title

In [ ]:
labels_titles = hdb.fit_predict(titles_emb.toarray())
n_clusters = len(set(labels_titles)) - (1 if -1 in labels_titles else 0)
n_noise = (labels_titles == -1).sum()
print(f"Found {n_clusters} clusters, and {n_noise} noisy points")


## Visualization

In [ ]:
now_labels = labels_self
now_unique_labels = sorted(set(now_labels), reverse=1)
labels_name = "HDBSCAN Titles"
labels_name = "Naive Categories"

In [ ]:

fig, axs = get_subplots(1,2, figsize=(12,6) )
fig2, axs2 = get_subplots(1,2, figsize=(12,6) )
lines = []
for index_label, label in enumerate(now_unique_labels):
    mask = now_labels == label
    color = colors[index_label] if label != -1 and label != "Technical Query / Other" else "grey"
    mk = "o" if label != -1 and label != "Technical Query / Other" else "x"


    axs[0].scatter(tsne_events[mask, 0], tsne_events[mask, 1], label=label, alpha=0.7, color=color, marker = mk)
    axs[1].scatter(pca_events[mask, 0], pca_events[mask, 1], label=label, alpha=0.7, color=color, marker = mk)

    axs2[0].scatter(tsne_titles[mask, 0], tsne_titles[mask, 1], label=label, alpha=0.7, color=color, marker = mk)
    axs2[1].scatter(pca_titles[mask, 0], pca_titles[mask, 1], label=label, alpha=0.7, color=color, marker = mk)
    lines.append(mlines.Line2D([], [], color=color, marker=mk, linestyle='None', label=label))


    print(f"Cluster : {label}")
    probs = []
    for index_name, name in enumerate(all_names):
        counts_faulty = X[now_labels == label, name_idx[name]] / 7
        counts_total = X_total[now_labels == label, name_idx[name]] / max_window[mask]

        counts_other_faulty = X[now_labels != label, name_idx[name]] / 7
        counts_other_total = X_total[now_labels != label, name_idx[name]] / max_window[~mask]

        stats, p = scipy.stats.mannwhitneyu(counts_faulty, counts_other_faulty, alternative="greater")

        probs.append([name, counts_faulty.mean(), counts_other_faulty.mean(), np.mean([c_f/(c_t+1) for c_f, c_t in zip(counts_faulty, counts_total)]), p, counts_total.mean(), counts_other_total.mean()])
        # if label == "Charging Failure" and name== "PowerConverter[X]":
        #     print(counts_faulty)
        #     print(counts_other_faulty)
        if p < 0.05:
            # print(counts_faulty, counts_total)
            # print(f"{name}: p={p:.4f} " + f"Mean Rate in cluster: {counts_faulty.mean()} vs other clusters: {counts_other_faulty.mean()}. Mean Relative Rate increase: {np.mean([c_f/(c_t+1) for c_f, c_t in zip(counts_faulty, counts_total)])}")
            pass
    probs = sorted(probs, key=lambda x: x[4], reverse=False)
    for i in probs[:3]:
        # print(i)
        print(f"name: {i[0]}, p-value: {i[4]:.4f} " + f"Mean Rate in cluster: {i[1]:.4f}/{i[5]:.4f} vs other clusters: {i[2]:.4f}/{i[6]:.4f}. Mean Relative Rate increase: {i[3]:.4f}")
    print("---\n")

fig.legend(handles=lines, title="Ticket Category", bbox_to_anchor=(.9, 0.9), loc='upper left')
fig2.legend(handles=lines, title="Ticket Category", bbox_to_anchor=(.9, 0.9), loc='upper left')

fig.suptitle(f"2D representaion of the event clusters\n defined by {labels_name} and using:", fontsize=fontsize,y=1.02)

axs[0].set_xlim(-40,40)
axs[0].set_ylim(-40,40)
fig2.suptitle(f"2D representaion of the titles clusters\n defined by {labels_name} and using:", fontsize=fontsize,y=1.02)

for ax in [axs, axs2]:
    ax[0].set_title("TSNE", fontsize=fontsize)
    ax[1].set_title("PCA", fontsize=fontsize)

for a in list(axs)+list(axs2):
    a.set_xlabel("x", fontsize=fontsize)
    a.set_ylabel("y", fontsize=fontsize)

In [ ]:
print("Popular vocab for clusters:")
for label in now_unique_labels:
    print(label, "--", end=" ")
    mask = now_labels == label

    mean_ = titles_emb[mask]
    mean_ = np.asarray(titles_emb[mask].mean(axis=0)).squeeze()
    top_indices = np.argsort(mean_)[::-1][:10]
    print(", ".join(vocab[top_indices]))


In [ ]:
fig, ax = get_subplots(figsize=(6,5))


ax.scatter(pca_events[:,0], pca_events[:,1], alpha=0.7, marker="o")

ax.set_xlabel("x", fontsize=fontsize)
ax.set_ylabel("y", fontsize=fontsize)
ax.set_title("2D projection of events logged \n 1 week prior to ticket", fontsize=fontsize)


ax.set_xlim(-1,1)
ax.set_ylim(-1,1)

plot.show()

# Full Ticket Information

In [ ]:
DATABASE_DIR = "../../Database"
TICKET_DIR = os.path.join(DATABASE_DIR, "Ticket_Extraction")
EVENT_DIR = os.path.join(DATABASE_DIR, "Event_Download")

fontsize = 18
if os.path.exists(DATABASE_DIR):
    print("Can see Database_Dir")

if os.path.exists(TICKET_DIR):
    print("Can see Ticket_Dir")

In [ ]:
charger_locations = pd.read_parquet(os.path.join(EVENT_DIR, "category_bucket_map.parquet"))

all_names = pd.read_parquet(os.path.join(EVENT_DIR, "all_names.parquet"))["name"].dropna().tolist()
min_max_events = pd.read_csv(os.path.join(DATABASE_DIR, "min_max_events.csv"))

sicharge_family = pd.read_csv(os.path.join(DATABASE_DIR, "SichargeD_Family_min_max.csv"))

## Embeddings

In [ ]:
summaries = []
names = []
charger_id = []
all_tickets   = os.path.join(TICKET_DIR, "Tickets_Trimmed_Summary")
for index, file_name in tqdm(enumerate(os.listdir(all_tickets))):
    names.append(file_name)
    with open(os.path.join(all_tickets, file_name), "r") as in_file:
        summaries.append(in_file.read())
    json_load = json.load(open(os.path.join(TICKET_DIR, "Tickets_Trimmed", file_name.split(".")[0]+".json"), "r"))
    charger_id.append([json_load["chargerID"], json_load["created_on"], json_load["incident_id"], int(index)])
    

charger_id = np.array(charger_id)

In [ ]:
embeddings_per_model = {}
for model_name in ["all-mpnet-base-v2", "paraphrase-mpnet-base-v2", "all-roberta-large-v1", "all-roberta-base-v1", "universal-sentence-encoder", "doc2vec", "allenai-specter"]:

    if model_name == "all-mpnet-base-v2":
        model = SentenceTransformer(os.path.join(DATABASE_DIR, "models", "all-mpnet-base-v2"))
    elif model_name == "paraphrase-mpnet-base-v2":
        model = SentenceTransformer(os.path.join(DATABASE_DIR, "models", "paraphrase-mpnet-base-v2"))

    elif model_name == "all-roberta-large-v1":
        model = SentenceTransformer(os.path.join(DATABASE_DIR, "models", "all-roberta-large-v1"))
    
    elif model_name == "allenai-specter":
        model = SentenceTransformer(os.path.join(DATABASE_DIR, "models", "allenai-specter"))

    elif model_name == "universal-sentence-encoder":
        model = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")
    elif model_name == "doc2vec":
        tagged_docs = [
            TaggedDocument(words=word_tokenize(summary.lower()), tags=[str(i)])
            for i, summary in enumerate(summaries)
        ]
        model = Doc2Vec(
            vector_size=100,
            min_count=2,
            epochs=40,
            workers=4
        )
        model.build_vocab(tagged_docs)
        model.train(tagged_docs, total_examples=model.corpus_count, epochs=model.epochs)

    if model_name != "universal-sentence-encoder" and model_name != "doc2vec":
        embeddings = model.encode(summaries, show_progress_bar=True)
    elif model_name == "universal-sentence-encoder":
        embeddings = model(summaries).numpy()
    elif model_name == "doc2vec":
        embeddings = np.array([model.dv[str(i)] for i in range(len(summaries))])

    # labels = hdb.fit_predict(embeddings)
    # n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    # n_noise = list(labels).count(-1)

    # results[model_name] = {
    #     "n_clusters": n_clusters,
    #     "n_noise": n_noise,
    # }

    embeddings_per_model[model_name] = embeddings

In [ ]:
model_name = "allenai-specter"
embeddings = embeddings_per_model[model_name]


In [ ]:
# np.save(os.path.join(TICKET_DIR, "Ticket_Embeddings", f"{model_name}_embeddings.npy"), embeddings)


In [ ]:
embeddings_norm = sk.preprocessing.normalize(embeddings)

matrix = [embeddings_norm[i] @ embeddings_norm[j] for i in range(len(embeddings_norm)) for j in range(len(embeddings_norm))]
matrix = np.array(matrix).reshape(len(embeddings), len(embeddings))


In [ ]:
n_clusters = 6
k_means_model = sk.cluster.KMeans(n_clusters=n_clusters, random_state=42, n_init=20, max_iter=500).fit(embeddings_norm)
print(sk.metrics.silhouette_score(embeddings_norm, k_means_labels))

In [ ]:
# np.save(os.path.join(TICKET_DIR, "Ticket_Embeddings", f"{model_name}_k_means_labels.npy"), k_means_labels)

## Visualization

In [ ]:
pca_2d = sk.decomposition.PCA(n_components=2)
tsne_2d = sk.manifold.TSNE(n_components=2, random_state=42)
pca = pca_2d.fit_transform(embeddings)
tsne = tsne_2d.fit_transform(embeddings)

In [ ]:
fig, ax  = get_subplots()
ax_twinx = ax.twinx()

fig_t, ax_t  = get_subplots(1,2, figsize=(18, 6))
ax2 = ax_t[0]
ax3 = ax_t[1]
ax2_twinx = ax2.twinx()
ax3_twinx = ax3.twinx()


prepare_subplots([ax_twinx, ax2_twinx, ax3_twinx], GRID=0)
ax.plot(list(results.keys()), [res[0] for res in results.values()], label="Silhouette Score")
ax_twinx.plot(list(results.keys()), [res[1] for res in results.values()], label="Number of Clusters", color=color_dic["default"][1])

ax.set_xlabel("Number of Clusters", fontsize=17)
ax.set_ylabel("Silhouette Score", fontsize=17, color=color_dic["default"][0])
ax_twinx.set_ylabel("davies_bouldin_cos", fontsize=17, color=color_dic["default"][1])
ax.set_title("Silhouette Score for KMeans Clustering", fontsize=17)

ax.set_ylim(0, 0.12)
ax3.set_ylim

fig_t.suptitle("Clustering based on similarity thresholds", fontsize=20)
ax2.plot(list(results_th.keys()), [res[0] for res in results_th.values()], label="Silhouette Score")
ax2_twinx.plot(list(results_th.keys()), [res[1] for res in results_th.values()], label="Number of Clusters", color=color_dic["default"][1])
ax2.set_ylabel("Silhouette Score", color=color_dic["default"][0], fontsize=17)
ax2_twinx.set_ylabel("Number of Clusters", color=color_dic["default"][1], fontsize=17)
ax2.set_xlabel("Threshold", fontsize=17)
ax2.set_ylim(0,0.2)
ax2_twinx.set_ylim(0,50)
ax2.set_title("Silhouette Score and Number of Clusters \n for Different Thresholds", fontsize=17)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(0.04))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(0.04/5))

ax3.plot(list(results_th.keys()), [res[0] for res in results_th.values()], label="Silhouette Score")
ax3_twinx.plot(list(results_th.keys()), [res[2] for res in results_th.values()], label="Davies-Bouldin Index", color=color_dic["default"][1])
ax3.set_ylabel("Silhouette Score", color=color_dic["default"][0], fontsize=17)
ax3_twinx.set_ylabel("Davies-Bouldin Index", color=color_dic["default"][1], fontsize=17)
ax3.set_xlabel("Threshold", fontsize=17)
ax3.set_ylim(0,0.2)
ax3_twinx.set_ylim(0,2)

fig_t.tight_layout()
plot.show()

In [ ]:
use_labels = k_means_labels
fig, ax = get_subplots(1,2,figsize=(12, 6))
scatter = ax[0].scatter(pca[:, 0], pca[:, 1], c=use_labels, cmap="tab20")
ax[0].set_title("PCA")
scatter = ax[1].scatter(tsne[:, 0], tsne[:, 1], c=use_labels, cmap="tab20")
ax[1].set_title("t-SNE")
handles, labels = scatter.legend_elements()
mapped_labels = [class_short_names.get(int(i), l) for i, l in enumerate(labels)]
legend = ax[1].legend(handles, mapped_labels, title="Class", loc=(1.05, 0.38))

ax[1].text(1.05, 0.28, f"Silhouette Score: {sk.metrics.silhouette_score(embeddings_norm, use_labels):.4f}", transform=ax[1].transAxes, fontsize=12, bbox=dict(facecolor='white', alpha=0.5))
fig.suptitle(f" Representation of embeddings of ticket summaries with KMeans Clustering (k={n_clusters})\n Using Small Dataset", fontsize=18, y=1.02)
# ax.legend()
plot.show()

In [ ]:
folder = os.path.join(TICKET_DIR, "Clusters")
create_tree_folder(folder, eraseLast=True)

for label_ix in set(k_means_labels):
    cluster_file = os.path.join(folder, f"Cluster_{label_ix}.txt")
    idx = np.where(use_labels == label_ix)[0]
    with open(cluster_file, "w") as out_file:
        for index in idx:
            out_file.write(summaries[index])
            out_file.write("-"*40)
            out_file.write("\n")

    
            sims = matrix[index, idx[idx != index]]  

